In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.lines import Line2D
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go
import plotly.colors as pc
import os
from scipy.sparse.linalg import eigs


In [ ]:
file_path = "/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam_full_check/weights/eigenvals.npz"

# Load the .npz file
loaded_data = np.load(file_path, allow_pickle=True)

# Convert to dictionary (if you want it as a standard Python dict)
eig = dict(loaded_data)

# Optional: check keys
print("Loaded keys:", eig.keys())


In [3]:
layers = [1,2]
gates = ['forget', 'cell', 'input', 'output']
weight_types = ['ih','hh']
eig_reshaped = {}
for layer in layers:
    for gate in gates:
        for weight_type in weight_types:
            eig_reshaped[f'layer{layer}_{gate}_gate_{weight_type}']= eig[f'layer{layer}_{gate}_gate_{weight_type}'].reshape(232,650)
    

In [5]:
xvals = []
for i in range(0,101,1):
    xvals.append(i/9269)
for i in range(200,9300,100):
    xvals.append(i/9269)
for i in range(1,41,1):
    xvals.append(i)
    
x_first_ep = []
for i in range(0,101,1):
    x_first_ep.append(i/9269)
for i in range(200,9300,100):
    x_first_ep.append(i/9269)
    
x_100_first_batches = []
for i in range(0,101,1):
    x_100_first_batches.append(i/9269)

In [16]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize

def plot_eig_scatter_through_time(eig, gates, xvals, layer, weight_type, top_n=10, figsize=(16, 14), layer_name = '40 epochs'):

    num_gates = len(gates)
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    axes = axes.flatten()
    norm = Normalize(vmin=min(xvals), vmax=max(xvals))
    for gate_idx, gate_name in enumerate(gates):
        ax = axes[gate_idx]
        for time_idx, time in enumerate(xvals):
            eig_vals = eig[f'layer{layer}_{gate_name}_gate_{weight_type}'][time_idx]
            magnitudes = np.abs(eig_vals)
            top_indices = np.argsort(magnitudes)[-top_n:]
            top_eig_vals = eig_vals[top_indices]

            # Plot the real vs imaginary scatter for top_n eigenvalues
            ax.scatter(np.real(top_eig_vals), np.imag(top_eig_vals), color=plt.cm.jet(time_idx / len(xvals)), label=f"Epoch {time_idx+1}")

        ax.set_title(f'{gate_name}', fontsize=14, fontweight='bold')
        ax.set_xlabel('Real Part')
        ax.set_ylabel('Imaginary Part')
        ax.grid(True, alpha=0.3)
        
        sm = plt.cm.ScalarMappable(cmap=plt.cm.jet, norm=norm)
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax, label='Epoch')
        
    fig.suptitle(f'Eigenvalue Evolution for Layer {layer} ({weight_type.upper()} Weights) ({layer_name})', fontsize=18, fontweight='bold')

    plt.tight_layout()
    plt.show()
    return fig


In [ ]:
plot_eig_scatter_through_time(eig_reshaped, gates, xvals, 1, 'hh', top_n=20, layer_name = '40 epochs')
plot_eig_scatter_through_time(eig_reshaped, gates, x_first_ep, 1, 'hh', top_n=20, layer_name = '1st epoch')
plot_eig_scatter_through_time(eig_reshaped, gates, x_100_first_batches, 1, 'hh', top_n=20, layer_name = '1st 100 batches')

In [ ]:
plot_eig_scatter_through_time(eig_reshaped, gates, xvals, 1, 'ih', top_n=20, layer_name = '40 epochs')
plot_eig_scatter_through_time(eig_reshaped, gates, x_first_ep, 1, 'ih', top_n=20, layer_name = '1st epoch')
plot_eig_scatter_through_time(eig_reshaped, gates, x_100_first_batches, 1, 'ih', top_n=20, layer_name = '1st 100 batches')

In [ ]:
plot_eig_scatter_through_time(eig_reshaped, gates, xvals, 2, 'hh', top_n=20, layer_name = '40 epochs')
plot_eig_scatter_through_time(eig_reshaped, gates, x_first_ep, 2, 'hh', top_n=20, layer_name = '1st epoch')
plot_eig_scatter_through_time(eig_reshaped, gates, x_100_first_batches, 2, 'hh', top_n=20, layer_name = '1st 100 batches')
plot_eig_scatter_through_time(eig_reshaped, gates, xvals, 2, 'ih', top_n=20, layer_name = '40 epochs')
plot_eig_scatter_through_time(eig_reshaped, gates, x_first_ep, 2, 'ih', top_n=20, layer_name = '1st epoch')
plot_eig_scatter_through_time(eig_reshaped, gates, x_100_first_batches, 2, 'ih', top_n=20, layer_name = '1st 100 batches')